# 1 · Run the detectors and the live web-page backend

This notebook does two things:
* **Try any detector directly** (image, document, audio, video, signature) with no web page - section 4.
* **Start the API with a public link** (ngrok) so the React page on your laptop can use it - sections 5 to 8.

**Use a GPU:** Runtime -> Change runtime type -> **T4 GPU**. Run the cells **top to bottom**.

What you need: the project zip (or a GitHub repo with it) and, for the web page, a free ngrok account (https://dashboard.ngrok.com -> *Your Authtoken*).

## 1. Install

In [ ]:
!pip install -q fastapi "uvicorn[standard]" python-multipart pyngrok requests
!pip install -q torch transformers pillow numpy pymupdf librosa soundfile opencv-python-headless scikit-image
print("Dependencies installed.")

## 2. Load the project code
Default: upload the project zip when asked. If the project is on GitHub, set `USE_ZIP_UPLOAD = False` and put your repository link in `REPO_URL`.

In [ ]:
import os, glob, subprocess, zipfile

USE_ZIP_UPLOAD = True                                            # False = clone from GitHub instead
REPO_URL = "https://github.com/YOUR_USERNAME/YOUR_REPO.git"      # only used when USE_ZIP_UPLOAD = False

if USE_ZIP_UPLOAD:
    from google.colab import files
    up = files.upload()
    with zipfile.ZipFile(list(up.keys())[0]) as z:
        z.extractall("/content/project")
    search_root = "/content/project"
else:
    if not os.path.exists("/content/repo"):
        subprocess.run(["git", "clone", "-q", REPO_URL, "/content/repo"], check=True)
    else:
        subprocess.run(["git", "-C", "/content/repo", "pull", "-q"], check=True)
    search_root = "/content/repo"

# Find the real project root (the folder that directly contains 07_api_backend)
hits = glob.glob(f"{search_root}/**/07_api_backend/main.py", recursive=True)
assert hits, "Could not find 07_api_backend/main.py - check the zip contents / REPO_URL."
SUITE_ROOT = os.path.dirname(os.path.dirname(hits[0]))
print("Suite root:", SUITE_ROOT)
print("Contents:", sorted(os.listdir(SUITE_ROOT)))

## 3. (Optional) Use your fine-tuned image model
Only if you trained one in **notebook 3**. Set `USE_FINETUNED = True`. The API and the detectors then use it for images and for video frames, and report its held-out test accuracy.
`FROM_DRIVE = True` reads `MyDrive/image_finetuned.zip` (faster); `False` asks you to upload the zip.

In [ ]:
USE_FINETUNED = False         # set True after notebook 3
FROM_DRIVE = True             # True = read MyDrive/image_finetuned.zip, False = upload it

if USE_FINETUNED:
    import shutil
    if FROM_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        zpath = "/content/drive/MyDrive/image_finetuned.zip"
    else:
        from google.colab import files
        up = files.upload()
        zpath = list(up.keys())[0]
    models_dir = os.path.join(SUITE_ROOT, "models")
    os.makedirs(models_dir, exist_ok=True)
    shutil.rmtree(os.path.join(models_dir, "image_finetuned"), ignore_errors=True)
    with zipfile.ZipFile(zpath) as z:
        z.extractall(models_dir)
    assert os.path.isfile(os.path.join(models_dir, "image_finetuned", "config.json")), "zip must contain an image_finetuned/ folder"
    print("Fine-tuned model installed:", os.path.join(models_dir, "image_finetuned"))
else:
    print("Using the original pretrained image model.")

## 4. (Optional) Try a detector directly - no web page needed
Pick the type in `MODALITY`, run the cell, and upload a file when asked. For **signature** you upload twice: first the known-genuine reference, then the signature to check.
The first run downloads the model, so it takes a minute or two.

In [ ]:
MODALITY = "image"   #@param ["image", "document", "audio", "video", "signature"]

import sys
from google.colab import files

for sub in ("01_image_detection", "02_document_detection", "03_audio_detection", "04_video_detection", "05_signature_detection"):
    p = os.path.join(SUITE_ROOT, sub)
    if p not in sys.path:
        sys.path.insert(0, p)
if SUITE_ROOT not in sys.path:
    sys.path.insert(0, SUITE_ROOT)
try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else None
except ImportError:
    DEVICE = None

def ask(prompt):
    print(prompt)
    return list(files.upload().keys())[0]

if MODALITY == "image":
    from image_detector import ImageDeepfakeDetector
    result = ImageDeepfakeDetector(device=DEVICE).predict(ask("Upload a photo (jpg/png):"))
elif MODALITY == "document":
    from doc_detector import DocumentDeepfakeDetector
    result = DocumentDeepfakeDetector(device=DEVICE).predict(ask("Upload a .pdf / .docx / .pptx:"))
elif MODALITY == "audio":
    from audio_detector import AudioDeepfakeDetector
    result = AudioDeepfakeDetector(device=DEVICE).predict(ask("Upload an audio clip (wav/mp3):"))
elif MODALITY == "video":
    from video_detector import VideoDeepfakeDetector
    result = VideoDeepfakeDetector(device=DEVICE, max_frames=8).predict(ask("Upload a SHORT video clip (mp4):"))
else:
    from signature_detector import SignatureDetector
    ref = ask("Upload the KNOWN-GENUINE reference signature:")
    questioned = ask("Upload the QUESTIONED signature to check:")
    result = SignatureDetector(device=DEVICE).compare(ref, questioned)

result.print_report()

## 5. ngrok credentials (for the web page)
Paste your authtoken when asked. `NGROK_DOMAIN`: leave `""` for a random link each session, or put your free static domain
(looks like `something-something.ngrok-free.dev`, find it under *Domains* on the ngrok dashboard) so the link never changes.
**Never paste your token into a file or share a screenshot of it.**

In [ ]:
from getpass import getpass

NGROK_AUTHTOKEN = getpass("Paste your ngrok authtoken (input hidden): ")
NGROK_DOMAIN = ""      # your static domain, or "" for a random one

## 6. Start the API server

In [ ]:
import subprocess, time, requests, os, sys

# Stop any previous server from an earlier run of this cell
try:
    server.terminate(); time.sleep(2)
except Exception:
    pass

env = {**os.environ, "DEEPFAKE_SUITE_ROOT": SUITE_ROOT, "CORS_ALLOW_ALL": "1"}  # demo: any frontend origin may call the API
server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "main:app", "--host", "0.0.0.0", "--port", "8000"],
    cwd=os.path.join(SUITE_ROOT, "07_api_backend"),
    env=env,
    stdout=open("/content/api.log", "w"),
    stderr=subprocess.STDOUT,
)

# Wait until it actually answers (instead of guessing with a fixed sleep)
for i in range(40):
    try:
        r = requests.get("http://localhost:8000/api/health", timeout=2)
        if r.status_code == 200:
            print("API is up locally:", r.json())
            break
    except Exception:
        pass
    time.sleep(1)
else:
    print("Server did not start. Last log lines:")
    print(open("/content/api.log").read()[-2000:])
    raise SystemExit

## 7. Open the public link

In [ ]:
from pyngrok import ngrok

ngrok.set_auth_token(NGROK_AUTHTOKEN)
for t in ngrok.get_tunnels():          # close stale tunnels from earlier runs
    ngrok.disconnect(t.public_url)

if NGROK_DOMAIN:
    tunnel = ngrok.connect(8000, domain=NGROK_DOMAIN)
else:
    tunnel = ngrok.connect(8000)

PUBLIC_URL = tunnel.public_url
print("=" * 64)
print("  YOUR LIVE API LINK:")
print("  ", PUBLIC_URL)
print("=" * 64)
print("Swagger docs :", PUBLIC_URL + "/docs")
print("Health check :", PUBLIC_URL + "/api/health")
print()
print("Put this in 08_frontend/.env on your laptop, then restart `npm run dev`:")
print(f"  VITE_API_URL={PUBLIC_URL}")

## 8. Check that the live link works
If the next cell fails with a "custom hostname" error, your `NGROK_DOMAIN` is wrong (it must be a domain from the ngrok dashboard, not a token). Set it to `""` and re-run sections 6-7.

In [ ]:
r = requests.get(PUBLIC_URL + "/api/health", headers={"ngrok-skip-browser-warning": "true"}, timeout=15)
print("Status:", r.status_code)
info = r.json()
print("Body  :", info)
assert r.status_code == 200, "Tunnel is up but the API is not answering through it."
print("\nLive link is working.")
print("-" * 64)
if info.get("image_model_is_finetuned"):
    print("Image model   : FINE-TUNED -", info.get("image_model"))
else:
    print("Image model   : !! ORIGINAL PRETRAINED MODEL !!")
    print("   It is close to a coin flip on this project's test data, so real photos can be called fake.")
    print("   Fix: set USE_FINETUNED = True in section 3 and run it, then re-run sections 6 and 7.")
if info.get("face_detection") is False:
    print("Face detection: !! OFF -", info.get("face_detection_reason"))
    print("   Fix: re-run the install cell (section 1), then Runtime -> Restart session, and run sections 2 to 7 again.")
else:
    print("Face detection: on")
print("-" * 64)
print("Open your React app now.")

## While it is running
**Server logs** (model-loading errors and bad uploads show up here):

In [ ]:
print(open("/content/api.log").read()[-4000:])

## Stop everything
Run this when you are done (or just disconnect the runtime).

In [ ]:
ngrok.kill()
server.terminate()
print("Server and tunnel stopped.")